In [106]:
from __future__ import annotations
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

In [107]:
ROOT = Path("/home/anastass/spoc-masked-attention/results/teacher-attention/iter_5000/lambda-scaling-test-kappa0p6")
OUT_DIR = Path("/home/anastass/spoc-masked-attention/results/teacher-attention/analysis/lambda-scaling-test-kappa0p2")
OUT_DIR.mkdir(parents=True, exist_ok=True)
AGG_PATH = OUT_DIR / "aggregated_lambda_sweep_all_rows.csv"

In [108]:
summary_files = sorted(ROOT.rglob("summary.csv"))
config_files = sorted(ROOT.rglob("config*.json"))

print("Number of summary.csv files:", len(summary_files))
print("Number of config json files:", len(config_files))

print("\nExample summary files:")
for p in summary_files[:5]:
    print(p)

print("\nExample config files:")
for p in config_files[:5]:
    print(p)

Number of summary.csv files: 9
Number of config json files: 78

Example summary files:
/home/anastass/spoc-masked-attention/results/teacher-attention/iter_5000/lambda-scaling-test-kappa0p6/maskrandom_r_120_rstar_120_sigstar_1_bstar_1_beta_1_d200_T5_lambda0p025_lr0p001_iter5000_pca120/maskrandom_r_120_rstar_120_sigstar_1_bstar_1_beta_1_d200_T5_lambda0p025_lr0p001_iter5000_pca120_seed_42_54081385/summary.csv
/home/anastass/spoc-masked-attention/results/teacher-attention/iter_5000/lambda-scaling-test-kappa0p6/maskrandom_r_120_rstar_120_sigstar_1_bstar_1_beta_1_d200_T5_lambda0p025_lr0p001_iter5000_pca120/maskrandom_r_120_rstar_120_sigstar_1_bstar_1_beta_1_d200_T5_lambda0p025_lr0p001_iter5000_pca120_seed_42_54081422/summary.csv
/home/anastass/spoc-masked-attention/results/teacher-attention/iter_5000/lambda-scaling-test-kappa0p6/maskrandom_r_120_rstar_120_sigstar_1_bstar_1_beta_1_d200_T5_lambda0p05_lr0p001_iter5000_pca120/maskrandom_r_120_rstar_120_sigstar_1_bstar_1_beta_1_d200_T5_lambda0p05

In [109]:
def flatten_dict(d: dict, parent_key: str = "", sep: str = ".") -> dict:
    out = {}
    for k, v in d.items():
        key = f"{parent_key}{sep}{k}" if parent_key else str(k)
        if isinstance(v, dict):
            out.update(flatten_dict(v, key, sep=sep))
        else:
            out[key] = v
    return out

In [110]:
def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [111]:
def infer_n_train(path: Path) -> int | None:
    text = str(path)
    m = re.search(r"ntrain[_-](\d+)", text)
    return int(m.group(1)) if m else None

In [112]:
def infer_seed(path: Path) -> int | None:
    text = str(path)
    m = re.search(r"seed[_-](\d+)", text)
    return int(m.group(1)) if m else None

In [113]:
def parse_config_folder_name(name: str) -> dict:
    """
    Parse metadata from folder names such as:
    maskrandom_r_25_rstar_25_sigstar_1_bstar_1_beta_1_d25_T5_lambda0p1_lr0p001_iter5000_pca25
    """
    def to_float(s: str | None):
        if s is None:
            return None
        return float(s.replace("p", "."))

    def grab(pattern: str):
        m = re.search(pattern, name)
        return m.group(1) if m else None

    return {
        "folder_config_name": name,
        "folder_mask_label": grab(r"^(mask[^_]+)"),
        "folder_r": int(grab(r"_r_(\d+)")) if grab(r"_r_(\d+)") else None,
        "folder_r_star": int(grab(r"_rstar_(\d+)")) if grab(r"_rstar_(\d+)") else None,
        "folder_d": int(grab(r"_d(\d+)")) if grab(r"_d(\d+)") else None,
        "folder_T": int(grab(r"_T(\d+)")) if grab(r"_T(\d+)") else None,
        "folder_lambda_reg": to_float(grab(r"_lambda([0-9p]+)")),
        "folder_lr": to_float(grab(r"_lr([0-9p]+)")),
        "folder_n_steps": int(grab(r"_iter(\d+)")) if grab(r"_iter(\d+)") else None,
        "folder_pca_n_components": int(grab(r"_pca(\d+)")) if grab(r"_pca(\d+)") else None,
    }

In [114]:
def get_top_config_folder(path: Path, root: Path = ROOT) -> Path:
    """
    For a path inside:
        ROOT / config_folder / job_folder / ntrain_folder / config.json

    return:
        ROOT / config_folder
    """
    rel = path.relative_to(root)
    return root / rel.parts[0]

In [115]:
rows = []

for config_path in config_files:
    try:
        config = load_json(config_path)
    except Exception as exc:
        print(f"[skip] could not read {config_path}: {exc}")
        continue

    flat = flatten_dict(config)

    top_config_folder = get_top_config_folder(config_path)
    run_folder = config_path.parent
    job_folder = run_folder.parent

    row = {
        "top_config_folder": str(top_config_folder),
        "top_config_folder_name": top_config_folder.name,
        "job_folder": str(job_folder),
        "run_folder": str(run_folder),
        "config_json": str(config_path),
        "inferred_n_train": infer_n_train(run_folder),
        "inferred_seed": infer_seed(run_folder),
        **parse_config_folder_name(top_config_folder.name),
        **flat,
    }

    rows.append(row)

config_df = pd.DataFrame(rows)

print(config_df.shape)
config_df.head()

(78, 55)


,top_config_folder,top_config_folder_name,job_folder,run_folder,config_json,inferred_n_train,inferred_seed,folder_config_name,folder_mask_label,folder_r,...,training.learning_rate,training.lambda_reg,evaluation.n_population,evaluation.eval_every,evaluation.track_attention_error_during_training,evaluation.attention_metric_subset_size,evaluation.pca_n_components,evaluation.n_random_baselines,logging.use_wandb,logging.project
0,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,1000,42,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,maskrandom,120,...,0.001,0.025,5000,25,True,512,120,10,False,spoc-masked-attention
1,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,100,42,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,maskrandom,120,...,0.001,0.025,5000,25,True,512,120,10,False,spoc-masked-attention
2,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,2000,42,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,maskrandom,120,...,0.001,0.025,5000,25,True,512,120,10,False,spoc-masked-attention
3,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,200,42,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,maskrandom,120,...,0.001,0.025,5000,25,True,512,120,10,False,spoc-masked-attention
4,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...,25,42,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,maskrandom,120,...,0.001,0.025,5000,25,True,512,120,10,False,spoc-masked-attention


In [116]:
def first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None

In [117]:
standard_cols = {
    "d": ["data.d", "folder_d"],
    "T": ["data.T", "folder_T"],
    "r": ["model.r", "folder_r"],
    "r_star": ["teacher.r_star", "folder_r_star"],
    "beta": ["model.beta"],
    "beta_star": ["teacher.beta_star"],
    "sigma_star": ["teacher.sigma_star"],
    "lambda_reg": ["training.lambda_reg", "folder_lambda_reg"],
    "learning_rate": ["training.learning_rate", "folder_lr"],
    "n_steps": ["training.n_steps", "folder_n_steps"],
    "n_train": ["training.n_train", "inferred_n_train"],
    "seed": ["experiment.seed", "experiment.master_seed", "inferred_seed"],
    "masking_strategy": ["data.masking_strategy"],
    "masks_per_sample": ["data.masks_per_sample"],
    "pca_n_components": ["evaluation.pca_n_components", "folder_pca_n_components"],
}

for new_col, candidates in standard_cols.items():
    col = first_existing(config_df, candidates)
    if col is not None:
        config_df[new_col] = config_df[col]
    else:
        config_df[new_col] = np.nan

In [118]:
numeric_cols = [
    "d", "T", "r", "r_star", "beta", "beta_star", "sigma_star",
    "lambda_reg", "learning_rate", "n_steps", "n_train", "seed",
    "masks_per_sample", "pca_n_components",
]

for c in numeric_cols:
    if c in config_df.columns:
        config_df[c] = pd.to_numeric(config_df[c], errors="coerce")

In [119]:
config_df["kappa"] = config_df["r"] / config_df["d"]
config_df["kappa_star"] = config_df["r_star"] / config_df["d"]
config_df["alpha_linear"] = config_df["n_train"] / config_df["d"]
config_df["alpha_quadratic"] = config_df["n_train"] / (config_df["d"] ** 2)
config_df["clt_baseline"] = 1.0 / np.sqrt(config_df["d"])
config_df["theoretical_psd_baseline"] = config_df["kappa_star"] / (1.0 + config_df["kappa_star"])

In [120]:
summary_rows = []

for summary_path in summary_files:
    try:
        df = pd.read_csv(summary_path)
    except Exception as exc:
        print(f"[skip] could not read {summary_path}: {exc}")
        continue

    if df.empty:
        print(f"[skip] empty summary: {summary_path}")
        continue

    top_config_folder = get_top_config_folder(summary_path)
    df = df.copy()

    df["top_config_folder"] = str(top_config_folder)
    df["top_config_folder_name"] = top_config_folder.name
    df["summary_csv"] = str(summary_path)
    df["summary_parent"] = str(summary_path.parent)

    summary_rows.append(df)

summary_df = pd.concat(summary_rows, ignore_index=True) if summary_rows else pd.DataFrame()

print(summary_df.shape)
summary_df.head()

(76, 79)


,alpha,master_seed,run_seed,seed,teacher_seed,train_data_seed,population_data_seed,student_init_seed,n_train,n_population,...,random_baseline_centered_cosine_S_S_star_mean,random_baseline_centered_cosine_S_S_star_std,random_baseline_cosine_S_S_star_mean,random_baseline_cosine_S_S_star_std,random_baseline_seed,ridge_lambda,top_config_folder,top_config_folder_name,summary_csv,summary_parent
0,NaN,42,2187595824,2187595824,42,2187595825,10000042,20000042,25,5000,...,-0.003689,0.008058,0.371793,0.005195,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...
1,NaN,42,933496318,933496318,42,933496319,10000042,20000042,100,5000,...,-0.003689,0.008058,0.371793,0.005195,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...
2,NaN,42,3832603222,3832603222,42,3832603223,10000042,20000042,200,5000,...,-0.003689,0.008058,0.371793,0.005195,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...
3,NaN,42,3013208154,3013208154,42,3013208155,10000042,20000042,500,5000,...,-0.003689,0.008058,0.371793,0.005195,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...
4,NaN,42,1658970725,1658970725,42,1658970726,10000042,20000042,1000,5000,...,-0.003689,0.008058,0.371793,0.005195,30000042,0.01,/home/anastass/spoc-masked-attention/results/t...,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...,/home/anastass/spoc-masked-attention/results/t...,/home/anastass/spoc-masked-attention/results/t...


In [121]:
summary_df.columns.tolist()

['alpha',
 'master_seed',
 'run_seed',
 'seed',
 'teacher_seed',
 'train_data_seed',
 'population_data_seed',
 'student_init_seed',
 'n_train',
 'n_population',
 'teacher_init',
 'r_star',
 'beta_star',
 'sigma_star',
 'train_loss',
 'population_risk',
 'generalization_gap',
 'ridge_train_loss',
 'ridge_population_risk',
 'ridge_generalization_gap',
 'attention_vs_ridge_gap',
 'attention_vs_ridge_relative_improvement',
 'pca_train_loss',
 'pca_population_risk',
 'pca_generalization_gap',
 'attention_vs_pca_gap',
 'attention_vs_pca_relative_improvement',
 'pca_n_components',
 'cosine_S_S_star',
 'random_baseline_cosine_S_S_star',
 'relative_error_S_S_star',
 'final_attention_level_error',
 'runtime_seconds',
 'runtime_per_step_seconds',
 'initial_objective',
 'final_objective',
 'best_objective',
 'objective_reduction',
 'initial_train_loss_history',
 'final_train_loss_history',
 'best_train_loss_history',
 'train_loss_reduction',
 'weight_norm',
 'W_star_norm',
 'S_trace',
 'S_top_eige

In [122]:
summary_n_col = first_existing(summary_df, ["n_train", "ntrain", "training.n_train"])
summary_seed_col = first_existing(summary_df, ["seed", "experiment.seed", "experiment.master_seed"])

if summary_n_col is None:
    raise ValueError("Could not find n_train column in summary_df.")

summary_df["n_train"] = pd.to_numeric(summary_df[summary_n_col], errors="coerce")

if summary_seed_col is not None:
    summary_df["seed"] = pd.to_numeric(summary_df[summary_seed_col], errors="coerce")
else:
    summary_df["seed"] = np.nan

In [123]:
config_merge = config_df.copy()
config_merge["top_config_folder"] = config_merge["top_config_folder"].astype(str)

summary_merge = summary_df.copy()
summary_merge["top_config_folder"] = summary_merge["top_config_folder"].astype(str)

summary_has_seed = summary_merge["seed"].notna().any()
config_has_seed = config_merge["seed"].notna().any()

if summary_has_seed and config_has_seed:
    merge_keys = ["top_config_folder", "n_train", "seed"]
else:
    merge_keys = ["top_config_folder", "n_train"]

print("Merge keys:", merge_keys)

aggregated = summary_merge.merge(
    config_merge,
    on=merge_keys,
    how="left",
    suffixes=("", "_config"),
)

print(aggregated.shape)
aggregated.head()

Merge keys: ['top_config_folder', 'n_train', 'seed']
(84, 152)


,alpha,master_seed,run_seed,seed,teacher_seed,train_data_seed,population_data_seed,student_init_seed,n_train,n_population,...,n_steps,masking_strategy,masks_per_sample,pca_n_components_config,kappa_config,kappa_star_config,alpha_linear,alpha_quadratic,clt_baseline,theoretical_psd_baseline
0,NaN,42,2187595824,2187595824,42,2187595825,10000042,20000042,25,5000,...,5000,random,1,120,0.6,0.6,0.125,0.000625,0.070711,0.375
1,NaN,42,2187595824,2187595824,42,2187595825,10000042,20000042,25,5000,...,5000,random,1,120,0.6,0.6,0.125,0.000625,0.070711,0.375
2,NaN,42,933496318,933496318,42,933496319,10000042,20000042,100,5000,...,5000,random,1,120,0.6,0.6,0.500,0.002500,0.070711,0.375
3,NaN,42,933496318,933496318,42,933496319,10000042,20000042,100,5000,...,5000,random,1,120,0.6,0.6,0.500,0.002500,0.070711,0.375
4,NaN,42,3832603222,3832603222,42,3832603223,10000042,20000042,200,5000,...,5000,random,1,120,0.6,0.6,1.000,0.005000,0.070711,0.375


In [124]:
aggregated.to_csv(AGG_PATH, index=False)
print("Saved:", AGG_PATH)

Saved: /home/anastass/spoc-masked-attention/results/teacher-attention/analysis/lambda-scaling-test-kappa0p2/aggregated_lambda_sweep_all_rows.csv


In [125]:
important_cols = [
    "d", "r", "r_star", "kappa", "kappa_star",
    "lambda_reg", "learning_rate", "n_train",
    "alpha_linear", "alpha_quadratic",
    "masking_strategy", "masks_per_sample",
    "top_config_folder_name",
]

available = [c for c in important_cols if c in aggregated.columns]
aggregated[available].head()

,d,r,r_star,kappa,kappa_star,lambda_reg,learning_rate,n_train,alpha_linear,alpha_quadratic,masking_strategy,masks_per_sample,top_config_folder_name
0,200,120,120,0.6,0.6,0.025,0.001,25,0.125,0.000625,random,1,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...
1,200,120,120,0.6,0.6,0.025,0.001,25,0.125,0.000625,random,1,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...
2,200,120,120,0.6,0.6,0.025,0.001,100,0.500,0.002500,random,1,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...
3,200,120,120,0.6,0.6,0.025,0.001,100,0.500,0.002500,random,1,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...
4,200,120,120,0.6,0.6,0.025,0.001,200,1.000,0.005000,random,1,maskrandom_r_120_rstar_120_sigstar_1_bstar_1_b...


In [126]:
cosine_candidates = [
    "cosine_S_S_star",
    "final_cosine_S_S_star",
    "teacher_cosine",
    "cosine_similarity",
]

cosine_col = None

for c in cosine_candidates:
    if c in aggregated.columns:
        cosine_col = c
        break

if cosine_col is None:
    possible = [
        c for c in aggregated.columns
        if "cosine" in c.lower()
        and "centered" not in c.lower()
        and "random" not in c.lower()
        and "baseline" not in c.lower()
    ]
    if possible:
        cosine_col = possible[0]

print("Using cosine column:", cosine_col)

Using cosine column: cosine_S_S_star


In [127]:
if cosine_col is None:
    raise ValueError("No learned non-centered cosine column found.")

aggregated["cosine"] = pd.to_numeric(aggregated[cosine_col], errors="coerce")

In [128]:
def summarize_lambda_group(g: pd.DataFrame, large_n_quantile: float = 0.7) -> pd.Series:
    g = g.dropna(subset=["n_train", "cosine"]).sort_values("n_train")

    n_threshold = g["n_train"].quantile(large_n_quantile)
    g_large = g[g["n_train"] >= n_threshold]

    final_row = g.loc[g["n_train"].idxmax()]

    out = {
        "n_points": len(g),
        "n_min": g["n_train"].min(),
        "n_max": g["n_train"].max(),
        "large_n_threshold": n_threshold,
        "cosine_mean_all": g["cosine"].mean(),
        "cosine_median_all": g["cosine"].median(),
        "cosine_max": g["cosine"].max(),
        "cosine_std_all": g["cosine"].std(ddof=0),
        "cosine_mean_large_n": g_large["cosine"].mean(),
        "cosine_median_large_n": g_large["cosine"].median(),
        "cosine_final": final_row["cosine"],
        "n_train_final": final_row["n_train"],
    }

    optional_cols = [
        "train_loss",
        "population_risk",
        "generalization_gap",
        "relative_error_S_S_star",
        "attention_level_error",
        "top_eigenvalue",
        "top_eigenvalue_S",
        "S_top_eigenvalue",
        "trace_S",
        "effective_rank",
    ]

    for c in optional_cols:
        if c in g.columns:
            values = pd.to_numeric(g[c], errors="coerce")
            if values.notna().any():
                out[f"{c}_mean_large_n"] = values.loc[g_large.index].mean()
                out[f"{c}_final"] = values.loc[final_row.name]

    return pd.Series(out)

In [129]:
lambda_summary = (
    aggregated
    .dropna(subset=["d", "lambda_reg", "n_train", "cosine"])
    .groupby(["d", "lambda_reg"])
    .apply(summarize_lambda_group)
    .reset_index()
    .sort_values(["d", "lambda_reg"])
    .reset_index(drop=True)
)

lambda_summary_path = OUT_DIR / "lambda_summary_by_d.csv"
lambda_summary.to_csv(lambda_summary_path, index=False)

lambda_summary

,d,lambda_reg,n_points,n_min,n_max,large_n_threshold,cosine_mean_all,cosine_median_all,cosine_max,cosine_std_all,...,train_loss_mean_large_n,train_loss_final,population_risk_mean_large_n,population_risk_final,generalization_gap_mean_large_n,generalization_gap_final,relative_error_S_S_star_mean_large_n,relative_error_S_S_star_final,S_top_eigenvalue_mean_large_n,S_top_eigenvalue_final
0,25,0.025,10.0,25.0,40000.0,6500.0,0.617727,0.663856,0.888804,0.229296,...,0.254825,0.253379,0.254948,0.255028,0.000123,0.001648,0.588409,0.573602,1.896864,1.847748
1,25,0.050,10.0,25.0,40000.0,6500.0,0.618617,0.705401,0.829601,0.203830,...,0.262547,0.261006,0.262307,0.262362,-0.000240,0.001356,0.754131,0.752883,1.407468,1.356419
2,50,0.025,10.0,25.0,40000.0,6500.0,0.474438,0.417041,0.872380,0.261673,...,0.264913,0.264175,0.257987,0.258021,-0.006926,-0.006154,0.675960,0.633758,2.019658,1.936765
3,50,0.050,10.0,25.0,40000.0,6500.0,0.501833,0.505888,0.840751,0.262697,...,0.270694,0.269892,0.263488,0.263560,-0.007205,-0.006332,0.749745,0.736745,1.776119,1.785943
4,100,0.025,10.0,25.0,40000.0,6500.0,0.350990,0.279297,0.762597,0.215948,...,0.257772,0.259465,0.253322,0.253226,-0.004450,-0.006239,0.785072,0.723063,2.268777,1.864861
5,100,0.050,10.0,25.0,40000.0,6500.0,0.389608,0.318009,0.802232,0.246635,...,0.262070,0.263654,0.257459,0.257406,-0.004612,-0.006248,0.781330,0.748662,1.866355,1.777604
6,200,0.025,18.0,25.0,2000.0,470.0,0.108079,0.105667,0.181467,0.026182,...,0.233630,0.252482,0.301518,0.274578,0.067887,0.022097,1.732904,1.212580,23.994359,12.638262
7,200,0.050,6.0,25.0,2000.0,750.0,0.118754,0.108369,0.181744,0.035261,...,0.253374,0.256042,0.277089,0.274758,0.023715,0.018716,1.191766,1.107325,12.194765,9.949964


In [130]:
score_col = "cosine_median_large_n"

idx = lambda_summary.groupby("d")[score_col].idxmax()

best_lambda_by_d = (
    lambda_summary
    .loc[idx]
    .sort_values("d")
    .reset_index(drop=True)
)

best_path = OUT_DIR / "best_lambda_by_d.csv"
best_lambda_by_d.to_csv(best_path, index=False)

best_lambda_by_d[
    [
        "d",
        "lambda_reg",
        score_col,
        "cosine_mean_large_n",
        "cosine_final",
        "cosine_max",
        "n_points",
        "n_min",
        "n_max",
    ]
]

,d,lambda_reg,cosine_median_large_n,cosine_mean_large_n,cosine_final,cosine_max,n_points,n_min,n_max
0,25,0.025,0.876195,0.872724,0.888804,0.888804,10.0,25.0,40000.0
1,50,0.025,0.826013,0.805107,0.872380,0.872380,10.0,25.0,40000.0
2,100,0.050,0.733337,0.722688,0.802232,0.802232,10.0,25.0,40000.0
3,200,0.050,0.163959,0.163959,0.181744,0.181744,6.0,25.0,2000.0
